# Применимость клинических рекомендаций к протоколам осмотра

**AI Journey 2026 — медицинский кейс Сбера.** Решение обеих подзадач.

| | Метрика | Балл LB |
|---|---|---|
| Stage 1 — «Общий / Специальный» заголовок КР | macro-F0.5 ≈ 0.90 | 18.7 / 21 |
| Stage 2 — применимость заголовка к протоколу | macro-F2 ≈ 0.84 | 37.0 / 49 |
| **Итого** | | **55.7 / 70** |

Порядок ноутбука:
1. Окружение, сиды, поиск данных и модели
2. Метрика и подбор весов бленда и порога на OOF
3. Признаки и вспомогательные функции
4. Stage 1: TF-IDF + Calibrated LinearSVC и эмбеддинги RuBERT + LogisticRegression
5. Stage 2: 5-fold cross-encoder и CatBoost на эмбеддингах и клинических признаках
6. Сборка и проверка `submission.csv`

Все гиперпараметры, кроме весов бленда и порогов, заданы заранее. Веса и пороги подбираются **только на out-of-fold предсказаниях train**: тест в подборе не участвует.

## 0. Установка зависимостей

В Google Colab / Kaggle достаточно доустановить CatBoost (остальное уже есть в образе). Локально: `pip install -r requirements.txt`.

In [ ]:
import importlib.util
import subprocess
import sys

for pkg in ["catboost", "transformers", "sklearn"]:
    if importlib.util.find_spec(pkg) is None:
        name = "scikit-learn" if pkg == "sklearn" else pkg
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])

## 1. Окружение, сиды, данные и модель

Данные соревнования в репозиторий не входят. Положите `train_stage1.csv`, `test_stage1.csv`, `train_stage2.csv`, `test_stage2.csv` в папку `data/` рядом с ноутбуком (или подключите датасет на Kaggle) — путь находится автоматически.

Энкодер — `cointegrated/rubert-base-cased-nli-threeway` с Hugging Face; для офлайн-запуска можно положить его в `models/`.

In [ ]:
import copy
import gc
import os
import re
import sys
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from catboost import CatBoostClassifier

In [ ]:
# =====================================================================
# 1. КОНФИГУРАЦИЯ, ДЕВАЙСЫ И СИДЫ
# =====================================================================
RANDOM_STATE = 42

def seed_everything(seed: int = RANDOM_STATE):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

# Автоопределение вычислительного устройства
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    USE_CUDA = True
    SCALER = torch.amp.GradScaler("cuda")
    print(f"[+] Устройство: NVIDIA CUDA GPU ({torch.cuda.get_device_name(0)})")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    USE_CUDA = False
    SCALER = None
    print("[+] Устройство: Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    USE_CUDA = False
    SCALER = None
    print("[+] Устройство: CPU")

def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/aij-medical-data"),
        Path("/kaggle/input/aij-medical-data/data"),
        Path("/kaggle/input/ai-journey-2024-medical"),
        Path("/kaggle/input"),
        Path("data"),
        Path("../data"),
        Path("../../data"),
        Path("."),
    ]
    for c in candidates:
        if (c / "train_stage1.csv").is_file() and (c / "train_stage2.csv").is_file():
            print(f"[+] Данные найдены в директории: {c.resolve()}")
            return c.resolve()
            
    # Рекурсивный поиск по всем вложенным путям
    for root in [Path("/kaggle/input"), Path("."), Path("..")]:
        if root.exists():
            matches = list(root.glob("**/train_stage1.csv"))
            for m in matches:
                p = m.parent.resolve()
                if (p / "train_stage2.csv").is_file():
                    print(f"[+] Данные найдены через рекурсивный поиск: {p}")
                    return p
                    
    if Path("/kaggle/input").exists():
        print(f"[!] Содержимое /kaggle/input: {list(Path('/kaggle/input').iterdir())}")
    raise FileNotFoundError("Файлы датасета train_stage1.csv / train_stage2.csv не найдены!")

DATA_DIR = find_data_dir()
ROOT_DIR = Path(".").resolve()

def resolve_model_path(model_id: str) -> str:
    """Поддержка офлайн-запуска: проверка наличия локального датасета с моделью."""
    candidates = [
        Path(f"/kaggle/input/{model_id.replace('/', '-')}-model"),
        Path(f"/kaggle/input/{model_id.split('/')[-1]}"),
        Path(f"models/{model_id.split('/')[-1]}"),
        Path(model_id),
    ]
    for p in candidates:
        if p.exists() and (p / "config.json").exists():
            print(f"[+] Использована локальная модель: {p.resolve()}")
            return str(p.resolve())
    return model_id

BASE_TRANSFORMER = "cointegrated/rubert-base-cased-nli-threeway"
TRANSFORMER_MODEL = resolve_model_path(BASE_TRANSFORMER)

## 2. Метрика и подбор бленда на OOF

Официальные метрики асимметричны:
- **Stage 1 — macro-F0.5**: precision весит вдвое больше recall, лишняя «специальная» метка стоит дороже пропуска.
- **Stage 2 — macro-F2**: recall весит вчетверо больше, пропуск применимого раздела стоит дороже ложной тревоги.

Поэтому порог 0.5 не оптимален ни там, ни там. Для каждой подзадачи сеткой перебираются вес бленда двух моделей (шаг 0.05) и порог (0.10–0.90, шаг 0.01), максимизируя метрику на OOF-предсказаниях 5-fold CV. Найденные значения применяются к усреднённым по фолдам предсказаниям теста.

In [ ]:
# =====================================================================
# 2. МЕТРИКИ И АВТОМАТИЧЕСКАЯ КАЛИБРОВКА ПОРОГОВ (ЧЕСТНЫЙ OOF)
# =====================================================================
def compute_macro_fbeta(y_true: np.ndarray, y_pred: np.ndarray, beta: float) -> float:
    """Расчет соревновательной метрики Macro-F_beta по классам 0 и 1."""
    return float(fbeta_score(y_true, y_pred, beta=beta, average="macro", zero_division=0))

def optimize_blend_and_threshold(
    y_true: np.ndarray,
    oof_prob_a: np.ndarray,
    oof_prob_b: np.ndarray,
    test_prob_a: np.ndarray,
    test_prob_b: np.ndarray,
    beta: float,
    stage_name: str,
    step_w: float = 0.05,
    step_th: float = 0.01,
) -> Tuple[np.ndarray, np.ndarray, float, float, float]:
    """
    Полностью автономная оптимизация весов ансамбля и порога классификации.
    Подбор параметров выполняется ИСКЛЮЧИТЕЛЬНО на Out-Of-Fold (OOF) предсказаниях.
    Тестовая выборка не участвует в калибровке.
    """
    print(f"\n[+] Оптимизация блендинга и порога на OOF под Macro-F_{beta} ({stage_name})...")
    best_w = 0.5
    best_th = 0.5
    best_score = -1.0
    
    weights = np.linspace(0.0, 1.0, int(round(1.0 / step_w)) + 1)
    thresholds = np.linspace(0.10, 0.90, int(round(0.80 / step_th)) + 1)
    
    for w in weights:
        blend_oof = w * oof_prob_a + (1.0 - w) * oof_prob_b
        for th in thresholds:
            preds = (blend_oof >= th).astype(int)
            score = compute_macro_fbeta(y_true, preds, beta=beta)
            if score > best_score:
                best_score = score
                best_w = float(w)
                best_th = float(th)
                
    test_blend = best_w * test_prob_a + (1.0 - best_w) * test_prob_b
    test_preds = (test_blend >= best_th).astype(int)
    
    print(f"    [✓] Оптимальный вес модели A:     {best_w:.2f} (Модель B: {1.0 - best_w:.2f})")
    print(f"    [✓] Оптимальный порог отсечки:    {best_th:.3f}")
    print(f"    [✓] OOF Macro-F_{beta}:               {best_score:.4f}")
    
    return test_preds, test_blend, best_w, best_th, best_score

## 3. Признаки и вспомогательные функции

- `PairTextDataset` — датасет для cross-encoder с парной токенизацией (`title`, `protocol`) и `token_type_ids`.
- `extract_dense_embeddings` — mean pooling последнего слоя трансформера по маске внимания.
- `normalize_title` — нижний регистр и схлопывание пробелов (префикс до `;`, если он есть, отбрасывается).
- `extract_ovr_features` — 7 бинарных флагов подгрупп пациентов в заголовке: дети, пожилые, беременность/лактация, коморбидность, тяжёлое течение, хирургия, диагностика. Именно такие подгруппы по условию задачи делают раздел «Специальным».

In [ ]:
# =====================================================================
# 3. ВСПОМОГАТЕЛЬНЫЕ КЛАССЫ И ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ
# =====================================================================
class PairTextDataset(Dataset):
    """Датасет для парной классификации с явной поддержкой token_type_ids."""
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

def extract_dense_embeddings(texts: List[str], model, tokenizer, max_len: int = 128, batch_size: int = 32) -> np.ndarray:
    """Извлечение mean-pooling векторных представлений трансформера."""
    model.eval()
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            if USE_CUDA:
                with torch.amp.autocast("cuda"):
                    out = model(**enc)
            else:
                out = model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).expand(out.last_hidden_state.size()).float()
            sum_embs = torch.sum(out.last_hidden_state.float() * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            mean_pooled = (sum_embs / sum_mask).cpu().numpy()
            all_embs.append(mean_pooled)
    return np.vstack(all_embs)

def normalize_title(text: str) -> str:
    """Мягкая нормализация текста: удаление префикса нозологии при наличии точки с запятой."""
    parts = str(text).split(";", 1)
    p = parts[1] if len(parts) == 2 else parts[0]
    return " ".join(p.lower().split())

def extract_ovr_features(df: pd.DataFrame) -> np.ndarray:
    """Извлечение ключевых категориальных дескрипторов подпопуляций."""
    patterns = {
        "child": r"дет|педиатр|подрост|новорожд|младен|до 18 лет",
        "elderly": r"пожил|старш|гериатр|старше 65|старше 75",
        "preg": r"берем|гестац|лактац|груд.{0,3}вскарм",
        "comorbid": r"хбп|почечн|печёночн|печеночн|сахарн|диабет|вич|диализ",
        "severe": r"тяжёл|тяжел|резист|рефрактер|декомпенсац|осложн|шок",
        "surg": r"хирург|операц|резекц|трансплант|стентир|вмешательств",
        "diag": r"диагностик|жалоб|анамнез|физикальн|обследован"
    }
    feats = []
    for _, row in df.iterrows():
        t = str(row["title_text"]).lower()
        feats.append([float(bool(re.search(pat, t))) for pat in patterns.values()])
    return np.array(feats, dtype=np.float32)

## 4. Stage 1 — классификация заголовков КР

Две модели разной природы, обе в 5-fold `StratifiedKFold`:
- **TF-IDF + Calibrated LinearSVC.** Символьные 3–5-граммы (устойчивы к русской морфологии и медицинским сокращениям) и словные 1–2-граммы. Калибровка Платта (`CalibratedClassifierCV`) нужна, чтобы выход SVC был вероятностью и его можно было смешивать.
- **Эмбеддинги RuBERT (768-d) + 7 флагов подгрупп → StandardScaler + LogisticRegression.** Ловит семантику там, где нет точного совпадения слов.

Бленд и порог подбираются под macro-F0.5.

In [ ]:
# =====================================================================
# 4. STAGE 1: ПОЛНОЦЕННЫЙ ПАЙПЛАЙН (OOF КРОСС-ВАЛИДАЦИЯ И БЛЕНДИНГ)
# =====================================================================
def run_stage1_pipeline(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    tokenizer,
    dense_model
) -> Tuple[pd.DataFrame, np.ndarray]:
    print("\n" + "=" * 80)
    print(" 🔬 [STAGE 1] КЛАССИФИКАЦИЯ ЗАГОЛОВКОВ КР (5-FOLD OOF КРОСС-ВАЛИДАЦИЯ)")
    print("=" * 80)
    
    y_train = train_df["label"].values.astype(int)
    clean_tr = [normalize_title(t) for t in train_df["title_text"]]
    clean_te = [normalize_title(t) for t in test_df["title_text"]]
    
    print("[+] Извлечение трансформерных эмбеддингов RuBERT...")
    emb_tr = extract_dense_embeddings(clean_tr, dense_model, tokenizer, max_len=128)
    emb_te = extract_dense_embeddings(clean_te, dense_model, tokenizer, max_len=128)
    
    ovr_tr = extract_ovr_features(train_df)
    ovr_te = extract_ovr_features(test_df)
    
    X_dense_tr = np.hstack([emb_tr, ovr_tr])
    X_dense_te = np.hstack([emb_te, ovr_te])
    
    print("[+] 5-Fold кросс-валидация Stage 1 (TF-IDF Platt LinearSVC + Scaled LogisticRegression)...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    
    oof_svc = np.zeros(len(train_df), dtype=np.float32)
    oof_emb = np.zeros(len(train_df), dtype=np.float32)
    test_svc = np.zeros(len(test_df), dtype=np.float32)
    test_emb = np.zeros(len(test_df), dtype=np.float32)
    
    clean_tr_arr = np.array(clean_tr)
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(clean_tr_arr, y_train)):
        # 1. TF-IDF + Calibrated LinearSVC (dual=False для гарантированной совместимости)
        tfidf = FeatureUnion([
            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_features=12000)),
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=4000))
        ])
        X_tf_tr = tfidf.fit_transform(clean_tr_arr[tr_idx])
        X_tf_val = tfidf.transform(clean_tr_arr[val_idx])
        X_tf_te = tfidf.transform(clean_te)
        
        clf_svc = CalibratedClassifierCV(
            estimator=LinearSVC(C=0.5, dual=False, random_state=RANDOM_STATE),
            cv=3
        )
        clf_svc.fit(X_tf_tr, y_train[tr_idx])
        oof_svc[val_idx] = clf_svc.predict_proba(X_tf_val)[:, 1]
        test_svc += clf_svc.predict_proba(X_tf_te)[:, 1] / 5.0
        
        # 2. Scaled LogisticRegression над эмбеддингами
        pipe_emb = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
        )
        pipe_emb.fit(X_dense_tr[tr_idx], y_train[tr_idx])
        oof_emb[val_idx] = pipe_emb.predict_proba(X_dense_tr[val_idx])[:, 1]
        test_emb += pipe_emb.predict_proba(X_dense_te)[:, 1] / 5.0
        
    # Автономный подбор блендинга и порога на OOF под соревновательную метрику Macro-F0.5
    s1_preds, p_blend_s1, best_w_s1, best_th_s1, oof_f05 = optimize_blend_and_threshold(
        y_true=y_train,
        oof_prob_a=oof_emb,
        oof_prob_b=oof_svc,
        test_prob_a=test_emb,
        test_prob_b=test_svc,
        beta=0.5,
        stage_name="Stage 1",
    )
    
    df_s1 = pd.DataFrame({
        "stage": 1,
        "id": test_df["id"].values,
        "label": s1_preds
    })
    return df_s1, p_blend_s1

## 5. Stage 2 — применимость заголовка к протоколу

**Модель A — 5-fold cross-encoder** на том же RuBERT-NLI: пара `[CLS] заголовок [SEP] протокол`, 256 токенов, 3 эпохи, AdamW lr=2e-5. После каждой эпохи считается ROC-AUC на валидационном фолде, в конце фолда веса откатываются к лучшей эпохе.

**Модель B — CatBoost** на 1 548 признаках:
- клинические параметры из полного текста протокола: фракция выброса и возраст (пропуски заполняются медианой train, плюс флаги пропуска), упоминания аритмии, отёков, гипертензии;
- взаимодействие эмбеддингов заголовка и протокола: косинус, евклидово расстояние, поэлементное произведение и модуль разности.

Числовые параметры важны, потому что многие разделы КР отсекают пациентов по порогам (возраст, ФВ при ХСН), а трансформер с усечённым протоколом их часто не видит.

Бленд и порог подбираются под macro-F2.

In [ ]:
# =====================================================================
# 5. STAGE 2: 5-FOLD DEEP CROSS-ENCODER + МУЛЬТИМОДАЛЬНЫЙ CATBOOST
# =====================================================================
def extract_clean_protocol(protocol: str, max_chars: int = 1200) -> str:
    """Стандартное усечение длины протокола без добавления искусственных маркеров."""
    p = str(protocol).strip()
    return p[:max_chars] if len(p) > max_chars else p

def train_5fold_cross_encoder(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    epochs: int = 3
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Честное 5-Fold обучение Cross-Encoder с ПРАВИЛЬНОЙ парной токенизацией
    (text_a, text_b), поддержкой token_type_ids и сохранением лучшего чекпоинта.
    """
    print("[+] Обучение 5-Fold Deep Cross-Encoder (парная токенизация text_a, text_b)...")
    
    titles_tr = train_df["title_text"].tolist()
    protos_tr = [extract_clean_protocol(p) for p in train_df["protocol_text"]]
    
    titles_te = test_df["title_text"].tolist()
    protos_te = [extract_clean_protocol(p) for p in test_df["protocol_text"]]
    
    labels = train_df["label"].values.astype(int)
    
    ce_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    
    # Корректная парная токенизация теста: tokenizer(titles, protocols)
    test_enc = ce_tokenizer(
        titles_te,
        protos_te,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )
    test_loader = DataLoader(PairTextDataset(test_enc), batch_size=32, shuffle=False)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(train_df), dtype=np.float32)
    test_preds = np.zeros(len(test_df), dtype=np.float32)
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(titles_tr, labels)):
        print(f"    -> Фолд {fold + 1} / 5...")
        
        tr_titles = [titles_tr[i] for i in tr_idx]
        tr_protos = [protos_tr[i] for i in tr_idx]
        val_titles = [titles_tr[i] for i in val_idx]
        val_protos = [protos_tr[i] for i in val_idx]
        
        # Корректная парная токенизация: формирует корректные token_type_ids для сегментов A и B
        tr_enc = ce_tokenizer(tr_titles, tr_protos, truncation=True, padding=True, max_length=256, return_tensors="pt")
        val_enc = ce_tokenizer(val_titles, val_protos, truncation=True, padding=True, max_length=256, return_tensors="pt")
        
        tr_loader = DataLoader(PairTextDataset(tr_enc, labels[tr_idx]), batch_size=16, shuffle=True)
        val_loader = DataLoader(PairTextDataset(val_enc, labels[val_idx]), batch_size=32, shuffle=False)
        
        model = AutoModelForSequenceClassification.from_pretrained(
            TRANSFORMER_MODEL, num_labels=2, ignore_mismatched_sizes=True
        ).to(DEVICE)
        optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        
        best_val_score = -1.0
        best_model_weights = copy.deepcopy(model.state_dict())
        
        for epoch in range(epochs):
            model.train()
            for batch in tr_loader:
                optimizer.zero_grad()
                b_ids = batch["input_ids"].to(DEVICE)
                b_mask = batch["attention_mask"].to(DEVICE)
                b_lbl = batch["labels"].to(DEVICE)
                b_types = batch.get("token_type_ids", None)
                if b_types is not None:
                    b_types = b_types.to(DEVICE)
                
                if SCALER is not None:
                    with torch.amp.autocast("cuda"):
                        out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types, labels=b_lbl)
                        loss = out.loss
                    SCALER.scale(loss).backward()
                    SCALER.step(optimizer)
                    SCALER.update()
                else:
                    out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types, labels=b_lbl)
                    loss = out.loss
                    loss.backward()
                    optimizer.step()
                    
            # Валидация в конце эпохи для Early Stopping
            model.eval()
            epoch_val_probs = []
            with torch.no_grad():
                for batch in val_loader:
                    b_ids = batch["input_ids"].to(DEVICE)
                    b_mask = batch["attention_mask"].to(DEVICE)
                    b_types = batch.get("token_type_ids", None)
                    if b_types is not None:
                        b_types = b_types.to(DEVICE)
                    if USE_CUDA:
                        with torch.amp.autocast("cuda"):
                            out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                    else:
                        out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                    probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
                    epoch_val_probs.extend(probs)
                    
            val_auc = float(roc_auc_score(labels[val_idx], epoch_val_probs))
            if val_auc > best_val_score:
                best_val_score = val_auc
                best_model_weights = copy.deepcopy(model.state_dict())
                
        print(f"       [✓] Фолд {fold + 1} лучший валидационный ROC-AUC: {best_val_score:.4f}")
        
        # Откат к лучшим сохраненным весам
        model.load_state_dict(best_model_weights)
        model.eval()
        
        # Финальный расчет OOF предсказаний на лучшей модели фолда
        val_probs = []
        with torch.no_grad():
            for batch in val_loader:
                b_ids = batch["input_ids"].to(DEVICE)
                b_mask = batch["attention_mask"].to(DEVICE)
                b_types = batch.get("token_type_ids", None)
                if b_types is not None:
                    b_types = b_types.to(DEVICE)
                if USE_CUDA:
                    with torch.amp.autocast("cuda"):
                        out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                else:
                    out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
                val_probs.extend(probs)
        oof_preds[val_idx] = np.array(val_probs)
        
        # Инференс лучшей модели фолда на тестовую выборку
        fold_test_probs = []
        with torch.no_grad():
            for batch in test_loader:
                b_ids = batch["input_ids"].to(DEVICE)
                b_mask = batch["attention_mask"].to(DEVICE)
                b_types = batch.get("token_type_ids", None)
                if b_types is not None:
                    b_types = b_types.to(DEVICE)
                if USE_CUDA:
                    with torch.amp.autocast("cuda"):
                        out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                else:
                    out = model(b_ids, attention_mask=b_mask, token_type_ids=b_types)
                probs = torch.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
                fold_test_probs.extend(probs)
        test_preds += np.array(fold_test_probs) / 5.0
        
        del model, optimizer, tr_loader, val_loader
        gc.collect()
        if USE_CUDA:
            torch.cuda.empty_cache()
            
    return oof_preds, test_preds

def extract_clinical_features_without_hardcode(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Честное извлечение числовых параметров протоколов:
    - Расчет медиан строго на train выборке (без утечки данных).
    - Флаги пропусков (missingness indicators) для дерева решений.
    - Без жестких эвристических вычитаний констант (-40, 18, 60).
    """
    def parse_values(df: pd.DataFrame):
        ef_list, age_list = [], []
        arrh_list, edem_list, hyper_list = [], [], []
        for _, row in df.iterrows():
            proto = str(row["protocol_text"]).lower()
            m_ef = re.search(r"(?:фв|фракци.{1,6}выброс[а-я]*)\s*[:=–—\-]?\s*(\d{2})", proto)
            m_age = re.search(r"(\d{1,3})\s*(?:лет|год|года|г\.)", proto)
            
            ef_list.append(float(m_ef.group(1)) if m_ef else np.nan)
            age_list.append(float(m_age.group(1)) if m_age else np.nan)
            arrh_list.append(float(bool(re.search(r"аритми|фибрилляц|трепетан|мерцательн", proto))))
            edem_list.append(float(bool(re.search(r"отек|отёк|пастозн", proto))))
            hyper_list.append(float(bool(re.search(r"гипертенз|гипертон|давлен", proto))))
            
        return pd.DataFrame({
            "ef": ef_list,
            "age": age_list,
            "arrhythmia": arrh_list,
            "edema": edem_list,
            "hypertension": hyper_list
        })

    tr_feat = parse_values(train_df)
    te_feat = parse_values(test_df)
    
    # Медианы обучающей выборки
    med_ef = float(tr_feat["ef"].median() if not np.isnan(tr_feat["ef"].median()) else 55.0)
    med_age = float(tr_feat["age"].median() if not np.isnan(tr_feat["age"].median()) else 58.0)
    
    for d in [tr_feat, te_feat]:
        d["ef_missing"] = d["ef"].isna().astype(float)
        d["age_missing"] = d["age"].isna().astype(float)
        d["ef"] = d["ef"].fillna(med_ef)
        d["age"] = d["age"].fillna(med_age)
        
    return tr_feat, te_feat

def compute_interaction_matrix(t_emb: np.ndarray, p_emb: np.ndarray) -> np.ndarray:
    """Семантическая матрица взаимодействия векторов заголовка и протокола."""
    dot = np.sum(t_emb * p_emb, axis=1, keepdims=True)
    norm_t = np.linalg.norm(t_emb, axis=1, keepdims=True) + 1e-9
    norm_p = np.linalg.norm(p_emb, axis=1, keepdims=True) + 1e-9
    cosine = dot / (norm_t * norm_p)
    euclid = np.linalg.norm(t_emb - p_emb, axis=1, keepdims=True)
    abs_diff = np.abs(t_emb - p_emb)
    elem_mult = t_emb * p_emb
    return np.hstack([cosine, euclid, elem_mult, abs_diff])

def run_stage2_pipeline(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    tokenizer,
    dense_model
) -> Tuple[pd.DataFrame, np.ndarray]:
    print("\n" + "=" * 80)
    print(" 🩺 [STAGE 2] ОЦЕНКА ПРИМЕНИМОСТИ КР (5-FOLD CROSS-ENCODER + CATBOOST OOF)")
    print("=" * 80)
    
    y_train = train_df["label"].values.astype(int)
    
    # 1. 5-Fold Deep Cross-Encoder
    oof_ce, test_ce = train_5fold_cross_encoder(train_df, test_df)
    
    # 2. Клинический парсинг и подготовка матриц взаимодействия
    print("[+] Извлечение клинических параметров протоколов...")
    feats_tr, feats_te = extract_clinical_features_without_hardcode(train_df, test_df)
    
    print("[+] Извлечение контекстных эмбеддингов заголовков и протоколов...")
    titles_tr = train_df["title_text"].tolist()
    protos_tr = [extract_clean_protocol(p) for p in train_df["protocol_text"]]
    t_emb_tr = extract_dense_embeddings(titles_tr, dense_model, tokenizer, max_len=128)
    p_emb_tr = extract_dense_embeddings(protos_tr, dense_model, tokenizer, max_len=256)
    inter_tr = compute_interaction_matrix(t_emb_tr, p_emb_tr)
    X_tr_s2 = np.hstack([feats_tr.values, inter_tr])
    
    titles_te = test_df["title_text"].tolist()
    protos_te = [extract_clean_protocol(p) for p in test_df["protocol_text"]]
    t_emb_te = extract_dense_embeddings(titles_te, dense_model, tokenizer, max_len=128)
    p_emb_te = extract_dense_embeddings(protos_te, dense_model, tokenizer, max_len=256)
    inter_te = compute_interaction_matrix(t_emb_te, p_emb_te)
    X_te_s2 = np.hstack([feats_te.values, inter_te])
    print(f"    Размерность матрицы признаков Stage 2: {X_tr_s2.shape}")
    
    # 3. 5-Fold кросс-валидация CatBoostClassifier
    print("[+] 5-Fold кросс-валидация CatBoostClassifier...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof_cb = np.zeros(len(train_df), dtype=np.float32)
    test_cb = np.zeros(len(test_df), dtype=np.float32)
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_s2, y_train)):
        cb = CatBoostClassifier(
            iterations=600,
            depth=5,
            learning_rate=0.04,
            l2_leaf_reg=3.0,
            auto_class_weights="SqrtBalanced",
            verbose=0,
            random_seed=RANDOM_STATE
        )
        cb.fit(
            X_tr_s2[tr_idx],
            y_train[tr_idx],
            eval_set=(X_tr_s2[val_idx], y_train[val_idx]),
            early_stopping_rounds=40,
            verbose=False
        )
        oof_cb[val_idx] = cb.predict_proba(X_tr_s2[val_idx])[:, 1]
        test_cb += cb.predict_proba(X_te_s2)[:, 1] / 5.0
        
    # Автономный подбор блендинга и порога на OOF под соревновательную метрику Macro-F2
    s2_preds, p_blend_s2, best_w_s2, best_th_s2, oof_f2 = optimize_blend_and_threshold(
        y_true=y_train,
        oof_prob_a=oof_ce,
        oof_prob_b=oof_cb,
        test_prob_a=test_ce,
        test_prob_b=test_cb,
        beta=2.0,
        stage_name="Stage 2",
    )
    
    df_s2 = pd.DataFrame({
        "stage": 2,
        "id": test_df["id"].values,
        "label": s2_preds
    })
    return df_s2, p_blend_s2

## 6. Запуск пайплайна

### 6.1. Данные и энкодер

In [ ]:
train_s1 = pd.read_csv(DATA_DIR / "train_stage1.csv").fillna("")
test_s1 = pd.read_csv(DATA_DIR / "test_stage1.csv").fillna("")
train_s2 = pd.read_csv(DATA_DIR / "train_stage2.csv").fillna("")
test_s2 = pd.read_csv(DATA_DIR / "test_stage2.csv").fillna("")

print(f"Stage 1: train = {len(train_s1)}, test = {len(test_s1)}, доля класса 1 = {train_s1['label'].mean():.3f}")
print(f"Stage 2: train = {len(train_s2)}, test = {len(test_s2)}, доля класса 1 = {train_s2['label'].mean():.3f}")

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
dense_model = AutoModel.from_pretrained(TRANSFORMER_MODEL).to(DEVICE)

### 6.2. Stage 1

In [ ]:
df_s1, p_s1 = run_stage1_pipeline(train_s1, test_s1, tokenizer, dense_model)

### 6.3. Stage 2

Самый долгий шаг: обучение пяти cross-encoder'ов. На GPU T4 — десятки минут, на CPU — несколько часов.

In [ ]:
df_s2, p_s2 = run_stage2_pipeline(train_s2, test_s2, tokenizer, dense_model)

### 6.4. Сборка и проверка `submission.csv`

In [ ]:
final_sub = pd.concat([df_s1, df_s2], ignore_index=True)
out_file = ROOT_DIR / "submission.csv"
final_sub.to_csv(out_file, index=False)

assert list(final_sub.columns) == ["stage", "id", "label"], "Неверные колонки сабмита!"
assert len(final_sub) == len(test_s1) + len(test_s2), "Число строк не совпадает с суммой тестов!"
assert final_sub["label"].isnull().sum() == 0, "Обнаружены NaN в колонке label!"
assert set(final_sub["label"].unique()).issubset({0, 1}), "В label есть значения кроме 0 и 1!"

s1_ones = int(final_sub[final_sub["stage"] == 1]["label"].sum())
s2_ones = int(final_sub[final_sub["stage"] == 2]["label"].sum())
print(f"Сохранено: {out_file}")
print(f"Stage 1: {s1_ones} / {len(test_s1)} единиц ({s1_ones / len(test_s1):.1%})")
print(f"Stage 2: {s2_ones} / {len(test_s2)} единиц ({s2_ones / len(test_s2):.1%})")

## 7. Ограничения и что дальше

- **Протокол усечён до 1 200 символов / 256 токенов** для трансформеров, а медиана длины протокола ~5 000 символов. Clinical-регулярки CatBoost смотрят полный текст, но семантика большей части протокола теряется. Следующий шаг — чанкинг протокола и агрегация, либо отбор релевантных заголовку фрагментов.
- **Условия заголовка не извлекаются явно.** Логичное развитие — разбирать заголовок на структурированные условия (возраст, пол, сопутствующее заболевание, тяжесть) и сопоставлять их с протоколом с учётом негаций («отрицает», «нет в анамнезе»).
- **Нормализация заголовка Stage 1** отбрасывает только префикс до `;`. Если название КР остаётся в тексте, модель может запоминать нозологии train; стоит проверить и вырезать название КР явно.
- **Медицинская доменная модель.** Энкодер — общий русскоязычный NLI-RuBERT; стоит сравнить с моделями, дообученными на медицинских текстах.